# RoBERTa-BiLSTM for Sentiment Analysis
**Based on**: Rahman et al., *"RoBERTa-BiLSTM: A Context-Aware Hybrid Model for Sentiment Analysis"*, IEEE TETCI, 2025

**Architecture**: RoBERTa (encoder) → Dropout → BiLSTM → Attention → Dense → Softmax

**Key idea**: RoBERTa generates contextual embeddings, BiLSTM captures long-range dependencies in those embeddings.

**Expected improvement over TextCNN (83.15%)**: RoBERTa understands context ("not good" ≠ "good") while BiLSTM reads the sequence of contextual embeddings.

In [1]:
"""
CELL 1 — CONFIGURATION
══════════════════════════════════════════════════════════
Hyperparameters from Paper 2 (Rahman et al., 2025):
  - Learning rate: 1e-5 (best across all their datasets)
  - Hidden units: 256 (best for BiLSTM)
  - Optimizer: AdamW
  - Epochs: 5
  - Dropout: 0.1 (between RoBERTa and BiLSTM)
"""

CONFIG = {
    # Data
    'DATA_PATH': '/kaggle/input/datasets/imuhammad/course-reviews-on-coursera/Coursera_reviews.csv',
    'SAMPLE_PER_CLASS': 30000,
    'MIN_WORDS': 8,
    'TEST_SIZE': 0.15,
    'VAL_SIZE': 0.15,
    'RANDOM_STATE': 42,
    
    # RoBERTa
    'MODEL_NAME': 'roberta-base',
    'MAX_LEN': 64,
    
    # BiLSTM (from Paper 2)
    'HIDDEN_DIM': 256,
    'NUM_LAYERS': 1,
    'BIDIRECTIONAL': True,
    
    # Training (from Paper 2)
    'BATCH_SIZE': 32,
    'EPOCHS': 6,
    'LEARNING_RATE': 2e-5,
    'DROPOUT': 0.2,
    'WEIGHT_DECAY': 0.01,
    'WARMUP_RATIO': 0.1,
    'EARLY_STOPPING_PATIENCE': 2,
    
    # Freezing Strategy
    'FREEZE_EMBEDDINGS': True,
    'FREEZE_N_LAYERS': 4,  # Freeze 4 of 12 RoBERTa layers, fine-tune top 8
    
    # Classes
    'NUM_CLASSES': 3,
    'CLASS_NAMES': ['Négatif', 'Neutre', 'Positif']
}

print('✅ Configuration loaded')
for k, v in CONFIG.items():
    print(f'   {k:<25}: {v}')

✅ Configuration loaded
   DATA_PATH                : /kaggle/input/datasets/imuhammad/course-reviews-on-coursera/Coursera_reviews.csv
   SAMPLE_PER_CLASS         : 30000
   MIN_WORDS                : 8
   TEST_SIZE                : 0.15
   VAL_SIZE                 : 0.15
   RANDOM_STATE             : 42
   MODEL_NAME               : roberta-base
   MAX_LEN                  : 64
   HIDDEN_DIM               : 256
   NUM_LAYERS               : 1
   BIDIRECTIONAL            : True
   BATCH_SIZE               : 32
   EPOCHS                   : 6
   LEARNING_RATE            : 2e-05
   DROPOUT                  : 0.2
   WEIGHT_DECAY             : 0.01
   WARMUP_RATIO             : 0.1
   EARLY_STOPPING_PATIENCE  : 2
   FREEZE_EMBEDDINGS        : True
   FREEZE_N_LAYERS          : 4
   NUM_CLASSES              : 3
   CLASS_NAMES              : ['Négatif', 'Neutre', 'Positif']


In [2]:
"""
CELL 2 — IMPORTS
══════════════════════════════════════════════════════════
"""

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'])

import warnings
warnings.filterwarnings('ignore')
import time
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, precision_score, recall_score)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

from transformers import RobertaTokenizer, RobertaModel

print('✅ All imports successful')
print(f'   PyTorch : {torch.__version__}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'   Device  : {device}')
if device.type == 'cuda':
    print(f'   GPU     : {torch.cuda.get_device_name(0)}')

✅ All imports successful
   PyTorch : 2.9.0+cu126
   Device  : cuda
   GPU     : Tesla T4


In [3]:

"""
CELL 3 — DATA LOADING + CLEANING
══════════════════════════════════════════════════════════
1. Remove non-English reviews (RoBERTa is English-only)
2. Filter short reviews (< MIN_WORDS)
3. Balance classes via stratified undersampling
"""

print("="*60)
print("🧹 DATA QUALITY IMPROVEMENTS")
print("="*60)

# ── Load data ──
df = pd.read_csv(CONFIG['DATA_PATH'])
if 'reviews' in df.columns:
    df = df.rename(columns={'reviews': 'Review', 'rating': 'Label'})
df['Review'] = df['Review'].fillna('').astype(str)

print(f"Raw dataset: {len(df):,} reviews")

# ── Improvement 1: Remove non-English reviews ──
print(f"\n📌 Improvement 1: Removing non-English reviews...")

# Simple but effective: check if review contains common English words
# Non-English reviews won't have many of these


def is_likely_english(text):
    """Remove only reviews that are CLEARLY non-English."""
    words = str(text).lower().split()
    if len(words) == 0:
        return False
    
    # Non-English indicators (Spanish, French, Portuguese, etc.)
    non_english_words = set([
        'el', 'la', 'los', 'las', 'es', 'un', 'una', 'del', 'que', 'en',
        'por', 'con', 'para', 'pero', 'como', 'más', 'muy', 'este', 'esta',
        'tiene', 'bien', 'malo', 'mejor', 'puede', 'hace', 'todo', 'también',
        'le', 'les', 'des', 'est', 'une', 'pas', 'dans', 'sur', 'avec',
        'sont', 'nous', 'vous', 'mais', 'cette', 'ces', 'qui', 'pour',
        'curso', 'bueno', 'malo', 'excelente', 'mucho', 'poco',
        'não', 'muito', 'bom', 'foi', 'isso', 'tem', 'uma',
    ])
    
    word_set = set(words)
    non_english_count = len(word_set.intersection(non_english_words))
    
    # Only remove if 30%+ of words are non-English
    ratio = non_english_count / len(words)
    # Note: 'es', 'en', 'un' exist in English too, but the 30% threshold
    # ensures we only remove reviews DOMINATED by non-English words
    return ratio < 0.3


before = len(df)
df['is_english'] = df['Review'].apply(is_likely_english)
df_english = df[df['is_english']].copy()
non_english_removed = before - len(df_english)
print(f"   Removed: {non_english_removed:,} non-English reviews")
print(f"   Remaining: {len(df_english):,}")

# Show some removed reviews
non_english_samples = df[~df['is_english']].sample(min(5, non_english_removed), random_state=42)
print(f"\n   Examples of removed reviews:")
for _, row in non_english_samples.iterrows():
    print(f"      \"{str(row['Review'])[:80]}...\"")

# ── Improvement 2: Increase min_words to 8 ──
print(f"\n📌 Improvement 2: Filtering reviews < 8 words...")

before2 = len(df_english)
df_english = df_english[df_english['Review'].str.split().str.len() >= CONFIG['MIN_WORDS']]
print(f"   Removed: {before2 - len(df_english):,} short reviews")
print(f"   Remaining: {len(df_english):,}")

# ── Apply sentiment labels ──
def label_to_sentiment(label):
    if label <= 2: return 0
    elif label == 3: return 1
    else: return 2

df_english['sentiment'] = df_english['Label'].apply(label_to_sentiment)

# ── Check pool sizes ──
print(f"\n📊 Available per class after cleaning:")
for i, name in enumerate(CONFIG['CLASS_NAMES']):
    count = (df_english['sentiment'] == i).sum()
    print(f"   {name:8} : {count:,}")

# ── Balanced sampling ──
dfs = []
for sentiment in range(CONFIG['NUM_CLASSES']):
    df_class = df_english[df_english['sentiment'] == sentiment]
    n = min(len(df_class), CONFIG['SAMPLE_PER_CLASS'])
    dfs.append(df_class.sample(n=n, random_state=CONFIG['RANDOM_STATE']))

df_balanced = pd.concat(dfs, ignore_index=True).sample(
    frac=1, random_state=CONFIG['RANDOM_STATE']
).reset_index(drop=True)

print(f"\n✅ Cleaned & balanced: {len(df_balanced):,} reviews")

# ── Split ──
X = df_balanced['Review'].values
y = df_balanced['sentiment'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=CONFIG['TEST_SIZE'] + CONFIG['VAL_SIZE'],
    stratify=y, random_state=CONFIG['RANDOM_STATE'])
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5,
    stratify=y_temp, random_state=CONFIG['RANDOM_STATE'])

print(f"   Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# ── Class weights ──
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

# ── Summary ──
print(f"""
{'═'*60}
📊 DATA IMPROVEMENTS SUMMARY
{'═'*60}
   Non-English removed  : {non_english_removed:,}
   Short reviews removed : {before2 - len(df_english):,} (< 8 words)
   Final balanced dataset: {len(df_balanced):,}
   
   Why this helps:
   • No Spanish/French reviews confusing English-only RoBERTa
   • Every review has enough words for meaningful context
   • Cleaner signal = better learning
{'═'*60}
""")

🧹 DATA QUALITY IMPROVEMENTS
Raw dataset: 1,454,711 reviews

📌 Improvement 1: Removing non-English reviews...
   Removed: 47,052 non-English reviews
   Remaining: 1,407,659

   Examples of removed reviews:
      "Bueno ..."
      "Es un curso donde te enseñan lo principal que debes saber del marketing. Te ayud..."
      "magnifico curso..."
      "excelente..."
      "Muy Interesante ..."

📌 Improvement 2: Filtering reviews < 8 words...
   Removed: 423,182 short reviews
   Remaining: 984,477

📊 Available per class after cleaning:
   Négatif  : 27,899
   Neutre   : 39,167
   Positif  : 917,411

✅ Cleaned & balanced: 87,899 reviews
   Train: 61,529 | Val: 13,185 | Test: 13,185

════════════════════════════════════════════════════════════
📊 DATA IMPROVEMENTS SUMMARY
════════════════════════════════════════════════════════════
   Non-English removed  : 47,052
   Short reviews removed : 423,182 (< 8 words)
   Final balanced dataset: 87,899
   
   Why this helps:
   • No Spanish/French review

In [4]:
"""
CELL 4 — ROBERTA TOKENIZER + DATASET
══════════════════════════════════════════════════════════

Key difference from DistilBERT notebook:
  - RoBERTa uses Byte-Pair Encoding (BPE) tokenizer, not WordPiece
  - RoBERTa uses <s> and </s> instead of [CLS] and [SEP]
  - RoBERTa was trained on 10x more data than BERT (160GB vs 16GB)
"""

print('='*60)
print('📝 ROBERTA TOKENIZATION')
print('='*60)

tokenizer = RobertaTokenizer.from_pretrained(CONFIG['MODEL_NAME'])
print(f'✅ Tokenizer loaded: {CONFIG["MODEL_NAME"]}')
print(f'   Vocabulary size: {tokenizer.vocab_size:,}')

# Demo
demo = "I don't recommend this course, it was not helpful at all."
tokens = tokenizer.tokenize(demo)
print(f'\n🔍 Demo:')
print(f'   Input : {demo}')
print(f'   Tokens: {tokens}')


class CourseraDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }


# Create datasets and loaders
train_dataset = CourseraDataset(X_train, y_train, tokenizer, CONFIG['MAX_LEN'])
val_dataset = CourseraDataset(X_val, y_val, tokenizer, CONFIG['MAX_LEN'])
test_dataset = CourseraDataset(X_test, y_test, tokenizer, CONFIG['MAX_LEN'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['BATCH_SIZE'])
test_loader = DataLoader(test_dataset, batch_size=CONFIG['BATCH_SIZE'])

print(f'\n✅ DataLoaders created:')
print(f'   Train: {len(train_loader)} batches')
print(f'   Val  : {len(val_loader)} batches')
print(f'   Test : {len(test_loader)} batches')

📝 ROBERTA TOKENIZATION


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: roberta-base
   Vocabulary size: 50,265

🔍 Demo:
   Input : I don't recommend this course, it was not helpful at all.
   Tokens: ['I', 'Ġdon', "'t", 'Ġrecommend', 'Ġthis', 'Ġcourse', ',', 'Ġit', 'Ġwas', 'Ġnot', 'Ġhelpful', 'Ġat', 'Ġall', '.']

✅ DataLoaders created:
   Train: 1923 batches
   Val  : 413 batches
   Test : 413 batches


In [5]:
"""
CELL 5 — ROBERTA-BILSTM-ATTENTION MODEL
══════════════════════════════════════════════════════════════
Based on TRABSA (94% F1) — adds attention layer after BiLSTM.

Architecture:
  RoBERTa → Dropout → BiLSTM → Attention → Dense → Softmax

Key change: Instead of using only the last BiLSTM hidden state,
attention learns WHICH tokens matter most for the decision.
"""

print('='*60)
print('🏗️ ROBERTA-BILSTM-ATTENTION MODEL')
print('='*60)


class SentimentAttention(nn.Module):
    """
    Attention mechanism that learns which BiLSTM outputs
    are most important for sentiment classification.
    
    Input:  BiLSTM outputs (batch, seq_len, hidden*2)
    Output: Weighted sum (batch, hidden*2)
    
    "I loved the content but hated the exams"
    → attention weights: loved(0.30) hated(0.35) but(0.15) ...
    → focuses on sentiment-bearing words
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1, bias=False)
        )
    
    def forward(self, lstm_output, mask=None):
        # lstm_output: (batch, seq_len, hidden_dim)
        # Compute attention scores
        scores = self.attention(lstm_output).squeeze(-1)  # (batch, seq_len)
        
        # Mask padded positions (set to -inf so softmax gives 0)
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))
        
        # Softmax → weights sum to 1
        weights = torch.softmax(scores, dim=1)  # (batch, seq_len)
        
        # Weighted sum of BiLSTM outputs
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)  # (batch, hidden_dim)
        
        return context, weights


class RoBERTaBiLSTMAttention(nn.Module):
    """
    RoBERTa-BiLSTM with Attention mechanism.
    Inspired by TRABSA (94% F1 on Twitter US Airline).
    
    RoBERTa: contextual token embeddings
    BiLSTM: sequential dependencies (all hidden states)
    Attention: learns which tokens matter most
    Dense: classification
    """
    def __init__(self, model_name, hidden_dim, num_classes, dropout,
                 num_layers=1, freeze_embeddings=True, freeze_n_layers=4):
        super().__init__()
        
        # ── RoBERTa ──
        self.roberta = RobertaModel.from_pretrained(model_name)
        roberta_hidden = self.roberta.config.hidden_size  # 768
        
        if freeze_embeddings:
            for param in self.roberta.embeddings.parameters():
                param.requires_grad = False
        if freeze_n_layers > 0:
            for i in range(freeze_n_layers):
                for param in self.roberta.encoder.layer[i].parameters():
                    param.requires_grad = False
        
        self.dropout = nn.Dropout(dropout)
        
        # ── BiLSTM ──
        self.bilstm = nn.LSTM(
            input_size=roberta_hidden,     # 768
            hidden_size=hidden_dim,         # 256
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        bilstm_dim = hidden_dim * 2  # 512
        
        # ── Attention (NEW — inspired by TRABSA) ──
        self.attention = SentimentAttention(bilstm_dim)
        
        # ── LayerNorm ──
        self.layer_norm = nn.LayerNorm(bilstm_dim)
        
        # ── Classification Head ──
        self.classifier = nn.Sequential(
            nn.Linear(bilstm_dim, hidden_dim),   # 512 → 256
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)    # 256 → 3
        )
    
    def forward(self, input_ids, attention_mask):
        # Step 1: RoBERTa
        roberta_output = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        sequence_output = roberta_output.last_hidden_state  # (batch, seq_len, 768)
        sequence_output = self.dropout(sequence_output)
        
        # Step 2: BiLSTM — get ALL hidden states (not just last)
        lengths = attention_mask.sum(dim=1).cpu().clamp(min=1)
        packed = nn.utils.rnn.pack_padded_sequence(
            sequence_output, lengths, batch_first=True, enforce_sorted=False
        )
        packed_output, _ = self.bilstm(packed)
        bilstm_output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output, batch_first=True
        )  # (batch, seq_len, 512)
        
        # Step 3: Attention — learn which tokens matter
        # Create boolean mask (True = valid token, False = padding)
        max_len = bilstm_output.size(1)
        bool_mask = torch.zeros(
            input_ids.size(0), max_len, device=input_ids.device, dtype=torch.bool
        )
        for i, length in enumerate(lengths):
            bool_mask[i, :length] = True
        
        context, attn_weights = self.attention(bilstm_output, mask=bool_mask)
        # context: (batch, 512) — weighted sum of all BiLSTM outputs
        
        # Step 4: LayerNorm + Classification
        context = self.layer_norm(context)
        logits = self.classifier(context)
        
        return logits


# ── Create model ──
model = RoBERTaBiLSTMAttention(
    model_name=CONFIG['MODEL_NAME'],
    hidden_dim=CONFIG['HIDDEN_DIM'],
    num_classes=CONFIG['NUM_CLASSES'],
    dropout=CONFIG['DROPOUT'],
    num_layers=CONFIG['NUM_LAYERS'],
    freeze_embeddings=CONFIG['FREEZE_EMBEDDINGS'],
    freeze_n_layers=CONFIG['FREEZE_N_LAYERS']
).to(device)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable

print(f'✅ RoBERTa-BiLSTM-Attention created')
print(f'   Total params    : {total:>12,}')
print(f'   Trainable params: {trainable:>12,} ({trainable/total:.1%})')
print(f'   Frozen params   : {frozen:>12,} ({frozen/total:.1%})')
print(f'\n   Architecture:')
print(f'   RoBERTa (freeze 0-{CONFIG["FREEZE_N_LAYERS"]-1})')
print(f'      ↓ (768-dim per token)')
print(f'   BiLSTM (hidden={CONFIG["HIDDEN_DIM"]}, bi → {CONFIG["HIDDEN_DIM"]*2})')
print(f'      ↓ (ALL hidden states, not just last)')
print(f'   Attention (learns which tokens matter)')
print(f'      ↓ (weighted sum → {CONFIG["HIDDEN_DIM"]*2}-dim)')
print(f'   Dense ({CONFIG["HIDDEN_DIM"]*2} → {CONFIG["HIDDEN_DIM"]} → {CONFIG["NUM_CLASSES"]})')

🏗️ ROBERTA-BILSTM-ATTENTION MODEL


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ RoBERTa-BiLSTM-Attention created
   Total params    :  127,143,171
   Trainable params:   59,791,107 (47.0%)
   Frozen params   :   67,352,064 (53.0%)

   Architecture:
   RoBERTa (freeze 0-3)
      ↓ (768-dim per token)
   BiLSTM (hidden=256, bi → 512)
      ↓ (ALL hidden states, not just last)
   Attention (learns which tokens matter)
      ↓ (weighted sum → 512-dim)
   Dense (512 → 256 → 3)


In [6]:
"""
CELL 6 — TRAINING SETUP WITH RLRP
══════════════════════════════════════════════════════════════
Uses ReduceLROnPlateau instead of fixed scheduler.
Based on ETRI paper (96.10% F1).
When val loss stops improving → LR is halved.
"""

print('='*60)
print('🔧 TRAINING SETUP (RLRP)')
print('='*60)

# Loss
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG['LEARNING_RATE'],
    weight_decay=CONFIG['WEIGHT_DECAY']
)

# RLRP Scheduler (from ETRI paper)
# When val_loss doesn't improve for 'patience' epochs → LR × factor
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # monitor val_loss (lower is better)
    factor=0.5,        # halve the LR
    patience=1,        # wait 1 epoch before reducing
    min_lr=1e-7,       # don't go below this
)

print(f'✅ Training setup:')
print(f'   Loss      : CrossEntropyLoss (weighted)')
print(f'   Optimizer : AdamW (lr={CONFIG["LEARNING_RATE"]})')
print(f'   Scheduler : ReduceLROnPlateau (factor=0.5, patience=1)')
print(f'   Logic     : if val_loss stalls for 1 epoch → LR ÷ 2')
print(f'   Min LR    : 1e-7')
print(f'   Optimizer LR: {optimizer.param_groups[0]["lr"]}')

🔧 TRAINING SETUP (RLRP)
✅ Training setup:
   Loss      : CrossEntropyLoss (weighted)
   Optimizer : AdamW (lr=2e-05)
   Scheduler : ReduceLROnPlateau (factor=0.5, patience=1)
   Logic     : if val_loss stalls for 1 epoch → LR ÷ 2
   Min LR    : 1e-7
   Optimizer LR: 2e-05


In [7]:
"""
CELL 7 — TRAINING LOOP (RLRP + ATTENTION)
══════════════════════════════════════════════════════════════
"""

print('='*60)
print('🚀 TRAINING RoBERTa-BiLSTM-Attention')
print('='*60)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = logits.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        if (batch_idx + 1) % 200 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f'      Batch {batch_idx+1}/{len(loader)} | '
                  f'Loss: {loss.item():.4f} | LR: {lr:.2e}')
    
    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader), acc, f1, all_preds, all_labels


# ═══════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════
best_val_f1 = 0
best_state = None
patience_counter = 0

start_time = time.time()

for epoch in range(CONFIG['EPOCHS']):
    print(f'\n{"─"*60}')
    print(f'   Epoch {epoch+1}/{CONFIG["EPOCHS"]} | LR: {optimizer.param_groups[0]["lr"]:.2e}')
    print(f'{"─"*60}')
    
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validate
    val_loss, val_acc, val_f1, _, _ = evaluate(
        model, val_loader, criterion, device
    )
    
    # RLRP: step scheduler with val_loss
    scheduler.step(val_loss)
    
    improved = '⭐' if val_f1 > best_val_f1 else ''
    print(f'\n   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%}')
    print(f'   Val Loss  : {val_loss:.4f} | Val Acc: {val_acc:.2%} | Val F1: {val_f1:.2%} {improved}')
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['EARLY_STOPPING_PATIENCE']:
            print(f'\n   ⏹️ Early stopping at epoch {epoch+1}')
            break

elapsed = time.time() - start_time

if best_state:
    model.load_state_dict(best_state)

print(f'\n{"═"*60}')
print(f'✅ Training complete in {elapsed:.1f}s ({elapsed/60:.1f} min)')
print(f'   Best Val F1: {best_val_f1:.2%}')
print(f'{"═"*60}')

🚀 TRAINING RoBERTa-BiLSTM-Attention

────────────────────────────────────────────────────────────
   Epoch 1/6 | LR: 2.00e-05
────────────────────────────────────────────────────────────
      Batch 200/1923 | Loss: 0.5984 | LR: 2.00e-05
      Batch 400/1923 | Loss: 0.5803 | LR: 2.00e-05
      Batch 600/1923 | Loss: 0.4272 | LR: 2.00e-05
      Batch 800/1923 | Loss: 0.4154 | LR: 2.00e-05
      Batch 1000/1923 | Loss: 0.5753 | LR: 2.00e-05
      Batch 1200/1923 | Loss: 0.4583 | LR: 2.00e-05
      Batch 1400/1923 | Loss: 0.2723 | LR: 2.00e-05
      Batch 1600/1923 | Loss: 0.5803 | LR: 2.00e-05
      Batch 1800/1923 | Loss: 0.3494 | LR: 2.00e-05

   Train Loss: 0.5671 | Train Acc: 75.48%
   Val Loss  : 0.5015 | Val Acc: 79.26% | Val F1: 79.35% ⭐

────────────────────────────────────────────────────────────
   Epoch 2/6 | LR: 2.00e-05
────────────────────────────────────────────────────────────
      Batch 200/1923 | Loss: 0.4398 | LR: 2.00e-05
      Batch 400/1923 | Loss: 0.4021 | LR: 2.0

In [8]:
"""
CELL 8 — FINAL EVALUATION + COMPARISON
══════════════════════════════════════════════════════════
"""

print('='*60)
print('📊 FINAL TEST EVALUATION')
print('='*60)

test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(
    model, test_loader, criterion, device
)

print(f'\n🎯 RoBERTa-BiLSTM Test Results:')
print(f'   Accuracy : {test_acc:.2%}')
print(f'   F1-Macro : {test_f1:.2%}')
print(f'   Precision: {precision_score(test_labels, test_preds, average="macro"):.2%}')
print(f'   Recall   : {recall_score(test_labels, test_preds, average="macro"):.2%}')

print(f'\n📋 Classification Report:')
print(classification_report(
    test_labels, test_preds,
    target_names=CONFIG['CLASS_NAMES'], digits=4
))

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
print(f'📊 Confusion Matrix:')
print(f'{"":>12} {"Pred Neg":>10} {"Pred Neu":>10} {"Pred Pos":>10}')
print(f'   {"─"*42}')
for i, name in enumerate(CONFIG['CLASS_NAMES']):
    row = cm[i]
    print(f'   {name:>8} | {row[0]:>8} {row[1]:>10} {row[2]:>10}  ({row[i]/row.sum():.0%} correct)')

# ═══════════════════════════════════════
# COMPARISON WITH ALL PREVIOUS MODELS
# ═══════════════════════════════════════
print(f'\n{"═"*70}')
print(f'📊 COMPLETE MODEL COMPARISON')
print(f'{"═"*70}')

print(f'''
   Phase 1: Random Embeddings (baseline)
   {'─'*60}
   {"Model":<30} {"F1-Macro":>12} {"Accuracy":>12}
   {'─'*60}
   {"LSTM (random 100d)":<30} {"24.07%":>12} {"35.73%":>12}
   {"GRU (random 100d)":<30} {"64.67%":>12} {"64.63%":>12}
   {"BiLSTM (random 100d)":<30} {"69.05%":>12} {"69.25%":>12}
   {"XGBoost (TF-IDF 5K)":<30} {"69.60%":>12} {"69.75%":>12}

   Phase 2: Pretrained Embeddings
   {'─'*60}
   {"DistilBERT (fine-tuned)":<30} {"77.20%":>12} {"77.21%":>12}
   {"FastText + FastText Clf":<30} {"78.58%":>12} {"78.62%":>12}
   {"FastText + BiLSTM-Fixed":<30} {"80.95%":>12} {"80.91%":>12}
   {"FastText + TextCNN":<30} {"83.15%":>12} {"83.14%":>12}

   Phase 3: RoBERTa Hybrid (this notebook)
   {'─'*60}
   {"RoBERTa-BiLSTM-Att":<30} {test_f1:>11.2%} {test_acc:>11.2%}  {"🏆" if test_f1 > 0.8315 else ""}
   {'─'*60}
''')

improvement = (test_f1 - 0.8315) * 100
print(f'   Improvement over TextCNN: {improvement:+.2f} points')
print(f'   Improvement over DistilBERT: {(test_f1 - 0.7720)*100:+.2f} points')


📊 FINAL TEST EVALUATION

🎯 RoBERTa-BiLSTM Test Results:
   Accuracy : 90.77%
   F1-Macro : 90.84%
   Precision: 90.92%
   Recall   : 90.86%

📋 Classification Report:
              precision    recall  f1-score   support

     Négatif     0.9113    0.9481    0.9294      4185
      Neutre     0.8629    0.8896    0.8760      4500
     Positif     0.9535    0.8882    0.9197      4500

    accuracy                         0.9077     13185
   macro avg     0.9092    0.9086    0.9084     13185
weighted avg     0.9092    0.9077    0.9079     13185

📊 Confusion Matrix:
               Pred Neg   Pred Neu   Pred Pos
   ──────────────────────────────────────────
    Négatif |     3968        200         17  (95% correct)
     Neutre |      319       4003        178  (89% correct)
    Positif |       67        436       3997  (89% correct)

══════════════════════════════════════════════════════════════════════
📊 COMPLETE MODEL COMPARISON
═════════════════════════════════════════════════════════════

In [11]:
"""
SAVE MODEL FOR DOWNLOAD
"""
import os, shutil

save_dir = '/kaggle/working/sentiment_roberta_bilstm'
os.makedirs(save_dir, exist_ok=True)

# Save model weights + config
torch.save({
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
    'test_f1': test_f1,
    'test_acc': test_acc,
    'architecture': 'RoBERTa-BiLSTM-Attention (91% F1)',
}, os.path.join(save_dir, 'roberta_bilstm_model.pkl'))

# Save tokenizer
tokenizer.save_pretrained(os.path.join(save_dir, 'tokenizer'))

# Create a ZIP for easy download
shutil.make_archive(
    '/kaggle/working/sentiment_roberta_bilstm',  # output name
    'zip',                                         # format
    '/kaggle/working',                             # root dir
    'sentiment_roberta_bilstm'                     # folder to zip
)

print(f'✅ Saved to {save_dir}')
for root, dirs, files in os.walk(save_dir):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1e6
        print(f'   {path} ({size:.1f} MB)')

print(f'\n📦 ZIP ready for download:')
print(f'   /kaggle/working/sentiment_roberta_bilstm.zip')
print(f'   Size: {os.path.getsize("/kaggle/working/sentiment_roberta_bilstm.zip") / 1e6:.1f} MB')

✅ Saved to /kaggle/working/sentiment_roberta_bilstm
   /kaggle/working/sentiment_roberta_bilstm/roberta_bilstm_model.pt (508.7 MB)
   /kaggle/working/sentiment_roberta_bilstm/roberta_bilstm_model.pkl (508.7 MB)
   /kaggle/working/sentiment_roberta_bilstm/tokenizer/tokenizer_config.json (0.0 MB)
   /kaggle/working/sentiment_roberta_bilstm/tokenizer/tokenizer.json (3.6 MB)

📦 ZIP ready for download:
   /kaggle/working/sentiment_roberta_bilstm.zip
   Size: 762.8 MB
